# LoRA Fine-Tuning: Code Generator

Fine-tuning `Qwen/Qwen2.5-0.5B` on the `Vezora/Tested-22k-Python-Alpaca` dataset using LoRA via PEFT and SFTTrainer from TRL.

**Goal:** Teach the base model to follow a structured instruction format and generate Python code with explanations and usage examples.

**Stack:** `transformers`, `peft`, `trl`, `datasets`, `accelerate`, `bitsandbytes`

**Hardware:** Google Colab T4 GPU (15GB VRAM)

**Adapter on HF Hub:** [Akram321/qwen2.5-code-generator-lora](https://huggingface.co/Akram321/qwen2.5-code-generator-lora)

## 1. Install Dependencies

In [1]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00


## 2. Login to Hugging Face Hub

Required to push the trained adapter at the end. Use a Write token from HF Settings.

In [3]:
from huggingface_hub import login
login()

## 3. Load and Inspect the Dataset

Dataset: `Vezora/Tested-22k-Python-Alpaca`
- 22,608 Python instruction-response pairs
- Columns: `instruction`, `input`, `output`
- Task: given a natural language instruction, generate a Python solution

In [4]:
from datasets import load_dataset
ds = load_dataset("Vezora/Tested-22k-Python-Alpaca")
print(ds)
print(ds["train"][0])

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

188k-Vezora-PyCode-Alpaca.json:   0%|          | 0.00/49.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22608 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 22608
    })
})
{'instruction': 'Create a nested loop to print every combination of numbers between 0-9, excluding any combination that contains the number 5. Additionally, exclude any combination that contains a repeating digit. Implement the solution without using any built-in functions or libraries to check for repeating digits.', 'input': '', 'output': 'Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:\n\n```python\nfor i in range(10):  # First digit\n    for j in range(10):  # Second digit\n        for k in range(10):  # Third digit\n            # Checking for the conditions\n            if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:\n                print(i, j, k)\n```\n\nThis code will generate and print every combination of three digit

## 4. Format the Dataset

Each sample is formatted into the Alpaca instruction template.
The resulting `text` column is what the model trains on.

Samples with a non-empty `input` field use the three-section format.
Samples with no `input` use the two-section format.

In [5]:
def format_sample(example):
  if example["input"]:
    text = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request
  ### Instruction:
  {example["instruction"]}

  ### Input:
  {example["input"]}

  ### Response
  {example["output"]}"""
  else:
    text = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.
  ### Instruction:
  {example["instruction"]}

  ### Response
  {example["output"]}
    """
  return {"text": text}

ds = ds.map(format_sample)
print(ds["train"][0]["text"])


Map:   0%|          | 0/22608 [00:00<?, ? examples/s]

Below is an instruction that describes a task. Write a response that appropriately completes the request.
  ### Instruction:
  Create a nested loop to print every combination of numbers between 0-9, excluding any combination that contains the number 5. Additionally, exclude any combination that contains a repeating digit. Implement the solution without using any built-in functions or libraries to check for repeating digits.

  ### Response 
  Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:

```python
for i in range(10):  # First digit
    for j in range(10):  # Second digit
        for k in range(10):  # Third digit
            # Checking for the conditions
            if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:
                print(i, j, k)
```

This code will generate and print every combination of three digits between 0-9 that do not conta

## 5. Load Model and Tokenizer

- `torch_dtype=torch.bfloat16`: loads weights in 16-bit to save memory
- `device_map="auto"`: places the model on GPU automatically
- `pad_token = eos_token`: Qwen2.5 has no dedicated pad token; we reuse EOS
- `padding_side = "right"`: pads on the right so real tokens stay left-aligned

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map ="auto"
 )

print(model)
print(f"Model loaded on: {model.device}")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 6. Apply LoRA Adapters via PEFT

LoRA injects small trainable matrices (rank `r=8`) next to the frozen `q_proj` and `v_proj` weight matrices in each of the 24 attention layers.

- `r=8`: rank of the low-rank decomposition
- `lora_alpha=16`: scaling factor (alpha = 2*r is the standard starting point)
- `target_modules`: which weight matrices get adapters (Q and V from the LoRA paper)
- Only ~0.11% of parameters are trainable — everything else is frozen

In [9]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.1 MB/s eta 0:00:00


In [10]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    target_modules = ["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 7. Train with SFTTrainer

SFTTrainer (Supervised Fine-Tuning Trainer) from TRL handles the full training loop.

Key settings:
- `fp16=True`: use 16-bit compute (required for T4, which does not support bf16)
- `per_device_train_batch_size=4` + `gradient_accumulation_steps=4`: effective batch size of 16
- `max_length=512`: truncate long samples to protect VRAM
- `num_train_epochs=1`: one full pass over 22,608 samples

In [16]:
from trl import SFTTrainer, SFTConfig
training_args = SFTConfig(
    output_dir="./qwen-lora-code",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="no",
    max_length=512,
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    processing_class=tokenizer,
)

trainer.train()

Building labels for train dataset:   0%|          | 0/22608 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.751295
40,0.651083
60,0.627634
80,0.608212
100,0.597722
120,0.593846
140,0.594727
160,0.594575
180,0.608165
200,0.599812


TrainOutput(global_step=1413, training_loss=0.5988438991646521, metrics={'train_runtime': 3835.6637, 'train_samples_per_second': 5.894, 'train_steps_per_second': 0.368, 'total_flos': 2.4560991104922624e+16, 'train_loss': 0.5988438991646521, 'entropy': 0.6316642606487641, 'num_tokens': 9627412.0, 'mean_token_accuracy': 0.8159721344709396, 'epoch': 1.0})

## 8. Inference: Fine-Tuned Model

Switch model to eval mode and re-enable the KV cache (disabled during training by gradient checkpointing).

In [19]:
model.config.use_cache = True
model = model.eval()

# disable gradient checkpointing for inference
model.gradient_checkpointing_disable()

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Write a Python function that takes a list of numbers and returns the maximum value without using the built-in max() function.

### Response:
"""

output = pipe(
    prompt,
    max_new_tokens=200,
    do_sample=False,
    temperature=None,
    top_p=None,
)
print(output[0]["generated_text"][len(prompt):])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```python
def find_max(numbers):
    max_value = numbers[0]
    for num in numbers:
        if num > max_value:
            max_value = num
    return max_value
```

This function takes a list of numbers as input and initializes the `max_value` variable with the first number in the list. It then iterates through the remaining numbers in the list and compares each number with the current `max_value`. If a number is greater than the current `max_value`, it updates the `max_value` variable. Finally, it returns the maximum value found.

Example usage:
```python
numbers = [1, 5, 2, 8, 3]
max_value = find_max(numbers)
print(max_value)  # Output: 8
```

In this example, the function is called with the list `[1, 5, 2, 8, 3]` as input. The maximum value in the list is `


## 9. Inference: Base Model (No LoRA)

Load the original base model with no adapter and run the same prompt to compare.

In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.float16,
    device_map="auto"
)
base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

base_pipe = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)

output_base = base_pipe(prompt, max_new_tokens=200, do_sample=False)
print(output_base[0]["generated_text"][len(prompt):])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```python
def find_max(numbers):
    max_value = numbers[0]
    for num in numbers:
        if num > max_value:
            max_value = num
    return max_value
```


## 10. Push Adapter to Hugging Face Hub

Only the LoRA adapter weights are pushed (~2.18MB), not the full base model.

In [20]:
model.push_to_hub("Akram321/qwen2.5-code-generator-lora")
tokenizer.push_to_hub("Akram321/qwen2.5-code-generator-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  26%|##5       |  557kB / 2.18MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpjl7tbb_7/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/Akram321/qwen2.5-code-generator-lora/commit/9496c2ca5c0968b8b7ad32b6f76c7302dd2051ce', commit_message='Upload tokenizer', commit_description='', oid='9496c2ca5c0968b8b7ad32b6f76c7302dd2051ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Akram321/qwen2.5-code-generator-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Akram321/qwen2.5-code-generator-lora'), pr_revision=None, pr_num=None)